Software Name: llm-acd-supplemental-material

SPDX-FileCopyrightText: Copyright (c) 2025 Orange SA

SPDX-License-Identifier: MIT


This software is distributed under the MIT license,
the text of which is available at https://opensource.org/license/MIT/
or see the "LICENSE" file for more details.


Authors: See CONTRIBUTORS.txt

Software description: The supplemental material for the paper From Zero-Shot to Reward-Aware: Evaluating Prompting and Memory in LLM-Based Cyber Defense Agents".

# Install

## Ecologits

In [ ]:
!pip install ecologits

## CybORG


In [ ]:
!git clone https://github.com/alan-turing-institute/CybORG_plus_plus.git

In [ ]:
!sed -i '114d' CybORG_plus_plus/Debugged_CybORG/CybORG/CybORG/Agents/Wrappers/TrueTableWrapper.py

In [ ]:
!pip install -e CybORG_plus_plus/Debugged_CybORG/CybORG
!pip install paramiko

In [ ]:
import os
import sys
# Replace with absolute path toward CybORG++ folder
sys.path.insert(0, os.path.abspath('/content/CybORG_plus_plus/Debugged_CybORG/CybORG/'))

# Import, Init & Utils

In [ ]:
import re
import time
import json
from tqdm.notebook import tqdm

from openai import AzureOpenAI
from ecologits import EcoLogits

from CybORG import CybORG
from CybORG.Agents.Wrappers import ChallengeWrapper
from CybORG.Agents.Wrappers.TrueTableWrapper import true_obs_to_table
from CybORG.Agents import B_lineAgent, RedMeanderAgent, GreenAgent, BaseAgent, SleepAgent

# LLM Setup

In [ ]:
# Replace with azure API endpoint
endpoint = ""

In [ ]:
# Replace with appropriate electricity zone
EcoLogits.init(providers="openai", electricity_mix_zone="SWE")

In [ ]:
# Enter you api key
azure_key = ""
client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=endpoint,
    api_key=azure_key,
)

# CybORG Setup

In [ ]:
def setup_scenario(scenario_path):
    # Get config
    with open(f'{scenario_path}.cfg', 'r') as file:
        scenario_cfg = file.read()
    scenario_cfg = json.loads(scenario_cfg)

    # List of actions
    actions = ['Analyse', 'Remove', 'DecoyApache', 'DecoyFemitter', 'DecoyHarakaSMPT',
    'DecoySmss', 'DecoySSHD', 'DecoySvchost', 'DecoyTomcat', 'DecoyVsftpd', 'Restore' ]

    # Get list of node labels from config
    labels = []
    labels.append('Defender')
    for enterprise in scenario_cfg['Enterprise']:
        labels.append(enterprise[0])
    for host in scenario_cfg['Op_Hosts']:
        labels.append(host)
    labels.append('Op_Server0')
    for user in scenario_cfg['User']:
        labels.append(user[0])

    # Construct an action/index map
    action_to_int = {
        'Sleep': 0,
        'Monitor': 1,
    }
    count = 2
    for action in actions:
        action_to_int[action] = {}
        for label in labels:
            action_to_int[action][label] = count
            count += 1

    # Construct Decoy Stg
    decoys = {
        'Defender': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'Enterprise0': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'Enterprise1': {
            'nb_decoy': 1,
            'decoys_lst': ['DecoyFemitter'],
            'tot_decoy': 1,
        },
        'Enterprise2': {
            'nb_decoy': 1,
            'decoys_lst': ['DecoyFemitter'],
            'tot_decoy': 1,
        },
        'Op_Host0': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'Op_Host1': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'Op_Host2': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'Op_Server0':{
            'nb_decoy': 4,
            'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
            'tot_decoy': 4,
        },
        'User0': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyTomcat', 'DecoyApache', 'DecoySmss', 'DecoySvchost'],
            'tot_decoy': 4,
        },
        'User1': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyTomcat', 'DecoyApache', 'DecoySmss', 'DecoySvchost'],
            'tot_decoy': 4,
        },
        'User2': {
            'nb_decoy': 4,
            'decoys_lst': ['DecoyFemitter', 'DecoyTomcat', 'DecoyApache', 'DecoySSHD'],
            'tot_decoy': 4,
        },
        'User3': {
            'nb_decoy': 2,
            'decoys_lst': ['DecoyVsftpd', 'DecoySSHD'],
            'tot_decoy': 2,
        },
        'User4': {
            'nb_decoy': 1,
            'decoys_lst': ['DecoyVsftpd'],
            'tot_decoy': 1,
        },
    }
    for label in labels:
        if label not in decoys:
            if 'User' in label:
                decoys[label] = {
                    'nb_decoy': 4,
                    'decoys_lst': ['DecoyTomcat', 'DecoyApache', 'DecoySmss', 'DecoySvchost'],
                    'tot_decoy': 4,
                }
            elif 'Enterprise' in label:
                decoys[label] = {
                    'nb_decoy': 4,
                    'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
                    'tot_decoy': 4,
                }
            elif 'Op_Host' in label:
                decoys[label] = {
                    'nb_decoy': 4,
                    'decoys_lst': ['DecoyVsftpd', 'DecoyHarakaSMPT', 'DecoyTomcat', 'DecoyApache'],
                    'tot_decoy': 4,
                }
            else:
                raise Exception("Invalid scenario configuration")

    return actions, labels, action_to_int, decoys, scenario_cfg

In [ ]:
def reset_decoys(decoys, target):
    decoys[target]['nb_decoy'] = decoys[target]['tot_decoy']
    return decoys

In [ ]:
def get_decoy(decoys, target):
    if decoys[target]['tot_decoy'] - decoys[target]['nb_decoy'] >= decoys[target]['tot_decoy']:
        raise Exception('No remaining decoy')
    decoy = decoys[target]['decoys_lst'][decoys[target]['tot_decoy'] - decoys[target]['nb_decoy']]
    decoys[target]['nb_decoy'] -= 1
    return decoy, decoys

In [ ]:
def get_env(scenario_path, seed, red_agent, green_agent=None, max_steps=128):
    agents = {'Red': red_agent,}

    if green_agent is not None:
        agents['Green'] = green_agent

    cyborg = CybORG(f'{scenario_path}.yaml', 'sim', agents = agents)
    env = ChallengeWrapper(env = cyborg, agent_name = 'Blue', max_steps = max_steps)
    env.set_seed(seed)

    return env, cyborg

In [ ]:
def format_obs(observation, labels, decoys, with_defender=True):
    obs_llm = "["
    start_index = 0 if with_defender else 4
    for node in range(start_index, len(observation) - 1, 4):
        match observation[node:node+2].tolist():
            case [0, 0]:
                activity = "None"
            case [1, 0]:
                activity = "Scan"
            case [1, 1]:
                activity = "Exploit"
            case _:
                raise Exception("Bad value activity")
        match observation[node+2:node+4].tolist():
            case [0, 0]:
                compromised = "No"
            case [0, 1]:
                compromised = "Unknown"
            case [1, 0]:
                compromised = "User"
            case [1, 1]:
                compromised = "Priviliged"
            case _:
                raise Exception("Bad compromised activity")
        obs_llm += f"({labels[node//4]}, {activity}, {compromised}, {decoys[labels[node//4]]['nb_decoy']}), "
    obs_llm = obs_llm[:-2]
    obs_llm += "]"
    return obs_llm

In [ ]:
def get_action_res(success):
    if success == 1:
        return 'Succeeded'
    elif success == 2:
        return 'Unknown'
    else:
        return 'Failed'

In [ ]:
def get_action_index(action, target, action_to_int):
    try:
        if target == '':
            return action_to_int[action]
        else:
            return action_to_int[action][target]
    except TypeError:
        raise Exception("Invalid action")

In [ ]:
def is_llm_response_valid(action, target, action_to_int):
    if action not in action_to_int:
        return False
    if action in ['Sleep', 'Monitor']:
        return target == ""
    return target in action_to_int[action]

# Logs & Evals

In [ ]:
def log_run(run_name, logs, prompt, params, scenario_cfg, path):
    content = prompt + "\n\n"
    content += "=" * 200 + "\n"
    content += "=" * 200 + "\n\n"
    content += str(params) + "\n\n"
    content += "=" * 200 + "\n"
    content += "=" * 200 + "\n\n"
    content += str(scenario_cfg) + "\n\n"
    content += "=" * 200 + "\n"
    content += "=" * 200 + "\n\n"
    content += logs + "\n"
    with open (path + run_name + ".log", 'w') as file:
        file.write(content)

# Parameters

In [ ]:
# LLM
model_name = "gpt-5"
temperature = 0
reasoning = "low"
max_tokens = 1024
params_llm = {
    "model_name": model_name,
    "temperature": temperature,
    "reasoning": reasoning,
    "max_tokens": max_tokens
}

# Logs
# Indicate where to save log
log_path = "llm_agents/logs/"

# Env
seed = 1
red_agent = RedMeanderAgent
green_agent = None
scenario_name = "base_scenario"
# Replace with your path
scenario_path = f"cyborg_scenari/{scenario_name}/{scenario_name}"
_, labels, action_to_int, decoys, scenario_cfg = setup_scenario(scenario_path)
env, cyborg = get_env(scenario_path, seed, red_agent, max_steps=128, green_agent=green_agent)

# Params
params = params_llm
params['red_agent'] = red_agent.__name__
if green_agent is not None:
    params['green_agent'] = green_agent.__name__
else:
    params['green_agent'] = 'None'

# Experiment - Single Step Prompt (Prompt 1 & 2)

In [ ]:
# Prompt
# Replace with your path
prompt_path = "llm_agents/prompts/"
prompt_file = "base-24-strat.md"
with open(prompt_path + prompt_file, "r") as file:
    system_prompt = file.read()

In [ ]:
nb_run = 15
num_steps = 100
global_run_name = "prompt2-base-BLine-noGreen-5"

for i in range(nb_run):
  print(f"Run number {i + 1}")

  # Run Params
  run_name = f"{global_run_name}-{i + 1}"
  print(run_name)

  # Init useful var
  regex = r'take_action\(([a-zA-Z]+), *(\w*)\)'
  logs = ""
  step = 0
  total_tokens = 0
  total_reasoning_tokens = 0
  total_reward = 0
  total_reward_30 = 0
  total_reward_50 = 0
  total_reward_100 = 0
  invalid_count = 0
  invalid_action = False

  # Init Env & action
  action = "None"
  observation, _ = env.reset()

  for i in tqdm(range(num_steps)):
      # Start turn with log
      step += 1
      logs += f"Step {step}\n"
      logs += f"Observation: {format_obs(observation, labels, decoys)}\n"
      logs += f"True State:\n{true_obs_to_table(cyborg.get_agent_state('True'), cyborg)}\n"

      # Start timer
      start = time.time()

      # Construct prompt
      prompt = [
          {
              "role": "system",
              "content": system_prompt,
          },
          {
              "role": "user",
              "content": format_obs(observation, labels, decoys),
          }
      ]

      # Get action from LLM
      if reasoning is None:
          response = client.chat.completions.create(
              messages=prompt,
              temperature=temperature,
              model=model_name,
              max_completion_tokens = max_tokens,
          )
      else:
          response = client.chat.completions.create(
              messages=prompt,
              reasoning_effort=reasoning,
              model=model_name,
              max_completion_tokens = max_tokens,
          )

      # Extract action and target
      try:
          regex_match = re.search(regex, response.choices[0].message.content)
          if regex_match:
              action = regex_match.group(1)
              target = regex_match.group(2)
          else:
              raise Exception
      except Exception:
          action = "Sleep"
          target = ""
          invalid_action = True
          invalid_count += 1

      # Handle Decoy
      if action == 'Decoy':
          try:
              action, decoys = get_decoy(decoys, target)
          except Exception:
              action = "Sleep"
              target = ""
              invalid_action = True
              invalid_count += 1
      if action == 'Restore':
          decoys = reset_decoys(decoys, target)

      # Check validity
      if not is_llm_response_valid(action, target, action_to_int):
          action = "Sleep"
          target = ""
          invalid_action = True
          invalid_count += 1

      # Get action index
      action_idx = get_action_index(action, target, action_to_int)

      # Take step
      observation, reward, _, _, _ = env.step(action_idx)

      # End timer
      end = time.time()

      # Log
      total_reward += reward
      total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
      if reasoning is not None:
          total_reasoning_tokens += response.usage.completion_tokens_details.reasoning_tokens
      logs += f"Red Action: {env.get_last_action('Red')} ({get_action_res(env.get_observation('Red')['success'].value)})\n"
      if invalid_action:
          logs += f"Blue Action (Invalid): {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
          invalid_action = False
      else:
          logs += f"Blue Action: {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
      logs += f"Reward: {reward}\n"
      logs += f"Prompt Tokens: {response.usage.prompt_tokens}\n"
      logs += f"Response Tokens: {response.usage.completion_tokens}\n"
      if reasoning is not None:
          logs += f"Reasoning Tokens: {response.usage.completion_tokens_details.reasoning_tokens}\n"
      logs += f"Response: {response.model_dump_json()}\n"
      logs += f"Time: {end - start} s\n"
      if step == 30:
          total_reward_30 = total_reward
      if step == 50:
          total_reward_50 = total_reward
      if step == 100:
          total_reward_100 = total_reward

      logs += "-"*200 + "\n"

  # Final log
  print(f"Reward at 30 steps: {total_reward_30}")
  print(f"Reward at 50 steps: {total_reward_50}")
  print(f"Reward at 100 steps: {total_reward_100}")
  print(f"Total reward: {total_reward}")
  print(f"Total invalid action: {invalid_count}")
  print(f"Total tokens: {total_tokens}")
  print(f"Total reasoning tokens: {total_reasoning_tokens}")
  logs += f"Reward at 30 steps: {total_reward_30}\n"
  logs += f"Reward at 50 steps: {total_reward_50}\n"
  logs += f"Reward at 100 steps: {total_reward_100}\n"
  logs += f"Total reward: {total_reward}\n"
  logs += f"Total invalid action: {invalid_count}\n"
  logs += f"Total tokens: {total_tokens}\n"
  logs += f"Total reasoning tokens: {total_reasoning_tokens}\n"
  logs += "-"*200 + "\n"
  logs += f"Final prompt: {prompt}\n"
  log_run(run_name, logs, system_prompt, params, scenario_cfg, log_path)

# Experiment - Autoregressive Prompt (Prompt 3 & 4)

In [ ]:
# Prompt
# Replace with your path
prompt_path = "llm_agents/prompts/"
prompt_file = "base-24-strat.md"
with open(prompt_path + prompt_file, "r") as file:
    system_prompt = file.read()

In [ ]:
nb_run = 15
num_steps = 100
global_run_name = "prompt4-base-BLine-noGreen-5"

for i in range(nb_run):
    print(f"Run number {i + 1}")

    # Run Params
    run_name = f"{global_run_name}-{i+1}"
    print(run_name)

    # Init useful var
    regex = r'take_action\(([a-zA-Z]+), *(\w*)\)'
    logs = ""
    step = 0
    total_tokens = 0
    total_reasoning_tokens = 0
    total_reward = 0
    total_reward_30 = 0
    total_reward_50 = 0
    total_reward_100 = 0
    invalid_count = 0
    invalid_action = False

    # Init Env & action
    action = "None"
    observation, _ = env.reset()

    # Init Prompt
    prompt = [
        {
            "role": "system",
            "content": system_prompt,
        }
    ]

    for i in tqdm(range(num_steps)):
        # Start turn with log
        step += 1
        logs += f"Step {step}\n"
        logs += f"Observation: {format_obs(observation, labels, decoys)}\n"
        logs += f"True State:\n{true_obs_to_table(cyborg.get_agent_state('True'), cyborg)}\n"

        # Start timer
        start = time.time()

        # Add observation to prompt
        prompt.append(
            {
                "role": "user",
                "content": format_obs(observation, labels, decoys),
            }
        )

        # Get action from LLM
        if reasoning is None:
            response = client.chat.completions.create(
                messages=prompt,
                temperature=temperature,
                model=model_name,
                max_completion_tokens = max_tokens,
            )
        else:
            response = client.chat.completions.create(
                messages=prompt,
                reasoning_effort=reasoning,
                model=model_name,
                max_completion_tokens = max_tokens,
            )

        # Extract action and target
        try:
            regex_match = re.search(regex, response.choices[0].message.content)
            if regex_match:
                action = regex_match.group(1)
                target = regex_match.group(2)
            else:
                raise Exception
        except Exception:
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Handle Decoy
        if action == 'Decoy':
            try:
                action, decoys = get_decoy(decoys, target)
            except Exception:
                action = "Sleep"
                target = ""
                invalid_action = True
                invalid_count += 1
        if action == 'Restore':
            decoys = reset_decoys(decoys, target)

        # Check validity
        if not is_llm_response_valid(action, target, action_to_int):
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Add LLM response to prompt history
        prompt.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content
            }
        )

        # Get action index
        action_idx = get_action_index(action, target, action_to_int)

        # Take step
        observation, reward, _, _, _ = env.step(action_idx)

        # End timer
        end = time.time()

        # Log
        total_reward += reward
        total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        if reasoning is not None:
            total_reasoning_tokens += response.usage.completion_tokens_details.reasoning_tokens
        logs += f"Red Action: {env.get_last_action('Red')} ({get_action_res(env.get_observation('Red')['success'].value)})\n"
        if invalid_action:
            logs += f"Blue Action (Invalid): {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
            invalid_action = False
        else:
            logs += f"Blue Action: {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
        logs += f"Reward: {reward}\n"
        logs += f"Prompt Tokens: {response.usage.prompt_tokens}\n"
        logs += f"Response Tokens: {response.usage.completion_tokens}\n"
        if reasoning is not None:
            logs += f"Reasoning Tokens: {response.usage.completion_tokens_details.reasoning_tokens}\n"
        logs += f"Response: {response.model_dump_json()}\n"
        logs += f"Time: {end - start} s\n"
        if step == 30:
            total_reward_30 = total_reward
        if step == 50:
            total_reward_50 = total_reward
        if step == 100:
            total_reward_100 = total_reward

        logs += "-"*200 + "\n"

    # Final log
    print(f"Reward at 30 steps: {total_reward_30}")
    print(f"Reward at 50 steps: {total_reward_50}")
    print(f"Reward at 100 steps: {total_reward_100}")
    print(f"Total reward: {total_reward}")
    print(f"Total invalid action: {invalid_count}")
    print(f"Total tokens: {total_tokens}")
    print(f"Total reasoning tokens: {total_reasoning_tokens}")
    logs += f"Reward at 30 steps: {total_reward_30}\n"
    logs += f"Reward at 50 steps: {total_reward_50}\n"
    logs += f"Reward at 100 steps: {total_reward_100}\n"
    logs += f"Total reward: {total_reward}\n"
    logs += f"Total invalid action: {invalid_count}\n"
    logs += f"Total tokens: {total_tokens}\n"
    logs += f"Total reasoning tokens: {total_reasoning_tokens}\n"
    logs += "-"*200 + "\n"
    logs += f"Final prompt: {prompt}\n"
    log_run(run_name, logs, system_prompt, params, scenario_cfg, log_path)

# Experiment - Autoregressive Prompt with Action feedback (Prompt 5 & 6)

In [ ]:
# Prompt
# Replace with your path
prompt_path = "llm_agents/prompts/"
prompt_file = "base-6-feedback-strat.md"
with open(prompt_path + prompt_file, "r") as file:
    system_prompt = file.read()

In [ ]:
nb_run = 15
num_steps = 100
global_run_name = "prompt6-base-BLine-noGreen-5"

for i in range(nb_run):
    print(f"Run number {i +1}")

    # Run Params
    run_name = f"{global_run_name}-{i+1}"
    print(run_name)

    # Init useful var
    regex = r'take_action\(([a-zA-Z]+), *(\w*)\)'
    logs = ""
    step = 0
    total_tokens = 0
    total_reasoning_tokens = 0
    total_reward = 0
    total_reward_30 = 0
    total_reward_50 = 0
    total_reward_100 = 0
    invalid_count = 0
    invalid_action = False

    # Init Env & action
    action = "None"
    observation, _ = env.reset()

    # Init Prompt
    prompt = [
        {
            "role": "system",
            "content": system_prompt,
        }
    ]

    for i in tqdm(range(num_steps)):
        # Start turn with log
        step += 1
        logs += f"Step {step}\n"
        logs += f"Observation: {format_obs(observation, labels, decoys)}\n"
        logs += f"True State:\n{true_obs_to_table(cyborg.get_agent_state('True'), cyborg)}\n"

        # Start timer
        start = time.time()

        # Add observation to prompt
        content = format_obs(observation, labels, decoys)
        if step != 1:
            content += '\n' + f'Last action: {get_action_res(env.get_observation('Blue')['success'].value)}.'
        else:
            content += '\n' + f'Last action: None.'
        prompt.append(
            {
                "role": "user",
                "content": content,
            }
        )

        # Get action from LLM
        if reasoning is None:
            response = client.chat.completions.create(
                messages=prompt,
                temperature=temperature,
                model=model_name,
                max_completion_tokens = max_tokens,
            )
        else:
            response = client.chat.completions.create(
                messages=prompt,
                reasoning_effort=reasoning,
                model=model_name,
                max_completion_tokens = max_tokens,
            )

        # Extract action and target
        try:
            regex_match = re.search(regex, response.choices[0].message.content)
            if regex_match:
                action = regex_match.group(1)
                target = regex_match.group(2)
            else:
                raise Exception
        except Exception:
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Handle Decoy
        if action == 'Decoy':
            try:
                action, decoys = get_decoy(decoys, target)
            except Exception:
                action = "Sleep"
                target = ""
                invalid_action = True
                invalid_count += 1
        if action == 'Restore':
            decoys = reset_decoys(decoys, target)

        # Check validity
        if not is_llm_response_valid(action, target, action_to_int):
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Add LLM response to prompt history
        prompt.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content
            }
        )

        # Get action index
        action_idx = get_action_index(action, target, action_to_int)

        # Take step
        observation, reward, _, _, _ = env.step(action_idx)

        # End timer
        end = time.time()

        # Log
        total_reward += reward
        total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        if reasoning is not None:
            total_reasoning_tokens += response.usage.completion_tokens_details.reasoning_tokens
        logs += f"Red Action: {env.get_last_action('Red')} ({get_action_res(env.get_observation('Red')['success'].value)})\n"
        if invalid_action:
            logs += f"Blue Action (Invalid): {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
            invalid_action = False
        else:
            logs += f"Blue Action: {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
        logs += f"Reward: {reward}\n"
        logs += f"Prompt Tokens: {response.usage.prompt_tokens}\n"
        logs += f"Response Tokens: {response.usage.completion_tokens}\n"
        if reasoning is not None:
            logs += f"Reasoning Tokens: {response.usage.completion_tokens_details.reasoning_tokens}\n"
        logs += f"Response: {response.model_dump_json()}\n"
        logs += f"Time: {end - start} s\n"
        if step == 30:
            total_reward_30 = total_reward
        if step == 50:
            total_reward_50 = total_reward
        if step == 100:
            total_reward_100 = total_reward

        #print("-"*200)
        logs += "-"*200 + "\n"

    # Final log
    print(f"Reward at 30 steps: {total_reward_30}")
    print(f"Reward at 50 steps: {total_reward_50}")
    print(f"Reward at 100 steps: {total_reward_100}")
    print(f"Total reward: {total_reward}")
    print(f"Total invalid action: {invalid_count}")
    print(f"Total tokens: {total_tokens}")
    print(f"Total reasoning tokens: {total_reasoning_tokens}")
    logs += f"Reward at 30 steps: {total_reward_30}\n"
    logs += f"Reward at 50 steps: {total_reward_50}\n"
    logs += f"Reward at 100 steps: {total_reward_100}\n"
    logs += f"Total reward: {total_reward}\n"
    logs += f"Total invalid action: {invalid_count}\n"
    logs += f"Total tokens: {total_tokens}\n"
    logs += f"Total reasoning tokens: {total_reasoning_tokens}\n"
    logs += "-"*200 + "\n"
    logs += f"Final prompt: {prompt}\n"
    log_run(run_name, logs, system_prompt, params, scenario_cfg, log_path)

# Experiment - Autoregressive Prompt with Reward feedback (Prompt 7 & 8)

In [ ]:
# Prompt
# Replace with your path
prompt_path = "llm_agents/prompts/"
prompt_file = "base-7-reward.md"
with open(prompt_path + prompt_file, "r") as file:
    system_prompt = file.read()

In [ ]:
nb_run = 15
num_steps = 100
global_run_name = "prompt7-base-BLine-noGreen-5"

for i in range(nb_run):
    print(f"Run number {i +1}")

    # Run Params
    run_name = f"{global_run_name}-{i+1}"
    print(run_name)

    # Init useful var
    regex = r'take_action\(([a-zA-Z]+), *(\w*)\)'
    logs = ""
    step = 0
    total_tokens = 0
    total_reasoning_tokens = 0
    total_reward = 0
    total_reward_30 = 0
    total_reward_50 = 0
    total_reward_100 = 0
    invalid_count = 0
    invalid_action = False

    # Init Env & action
    action = "None"
    observation, _ = env.reset()

    # Init Prompt
    prompt = [
        {
            "role": "system",
            "content": system_prompt,
        }
    ]

    for i in tqdm(range(num_steps)):
        # Start turn with log
        step += 1
        logs += f"Step {step}\n"
        logs += f"Observation: {format_obs(observation, labels, decoys)}\n"
        logs += f"True State:\n{true_obs_to_table(cyborg.get_agent_state('True'), cyborg)}\n"

        # Start timer
        start = time.time()

        # Add observation to prompt
        content = format_obs(observation, labels, decoys)
        if step != 1:
            content += '\n' + f'Last action: {get_action_res(env.get_observation('Blue')['success'].value)}.'
            content += '\n' + f'Reward: {reward}.'
        else:
            content += '\n' + f'Last action: None.'
        prompt.append(
            {
                "role": "user",
                "content": content,
            }
        )

        # Get action from LLM
        if reasoning is None:
            response = client.chat.completions.create(
                messages=prompt,
                temperature=temperature,
                model=model_name,
                max_completion_tokens = max_tokens,
            )
        else:
            response = client.chat.completions.create(
                messages=prompt,
                reasoning_effort=reasoning,
                model=model_name,
                max_completion_tokens = max_tokens,
            )

        # Extract action and target
        try:
            regex_match = re.search(regex, response.choices[0].message.content)
            if regex_match:
                action = regex_match.group(1)
                target = regex_match.group(2)
            else:
                raise Exception
        except Exception:
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Handle Decoy
        if action == 'Decoy':
            try:
                action, decoys = get_decoy(decoys, target)
            except Exception:
                action = "Sleep"
                target = ""
                invalid_action = True
                invalid_count += 1
        if action == 'Restore':
            decoys = reset_decoys(decoys, target)

        # Check validity
        if not is_llm_response_valid(action, target, action_to_int):
            action = "Sleep"
            target = ""
            invalid_action = True
            invalid_count += 1

        # Add LLM response to prompt history
        prompt.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content
            }
        )

        # Get action index
        action_idx = get_action_index(action, target, action_to_int)

        # Take step
        observation, reward, _, _, _ = env.step(action_idx)

        # End timer
        end = time.time()

        # Log
        total_reward += reward
        total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens
        if reasoning is not None:
            total_reasoning_tokens += response.usage.completion_tokens_details.reasoning_tokens
        logs += f"Red Action: {env.get_last_action('Red')} ({get_action_res(env.get_observation('Red')['success'].value)})\n"
        if invalid_action:
            logs += f"Blue Action (Invalid): {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
            invalid_action = False
        else:
            logs += f"Blue Action: {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
        logs += f"Reward: {reward}\n"
        logs += f"Prompt Tokens: {response.usage.prompt_tokens}\n"
        logs += f"Response Tokens: {response.usage.completion_tokens}\n"
        if reasoning is not None:
            logs += f"Reasoning Tokens: {response.usage.completion_tokens_details.reasoning_tokens}\n"
        logs += f"Response: {response.model_dump_json()}\n"
        logs += f"Time: {end - start} s\n"
        if step == 30:
            total_reward_30 = total_reward
        if step == 50:
            total_reward_50 = total_reward
        if step == 100:
            total_reward_100 = total_reward

        logs += "-"*200 + "\n"

    # Final log
    print(f"Reward at 30 steps: {total_reward_30}")
    print(f"Reward at 50 steps: {total_reward_50}")
    print(f"Reward at 100 steps: {total_reward_100}")
    print(f"Total reward: {total_reward}")
    print(f"Total invalid action: {invalid_count}")
    print(f"Total tokens: {total_tokens}")
    print(f"Total reasoning tokens: {total_reasoning_tokens}")
    logs += f"Reward at 30 steps: {total_reward_30}\n"
    logs += f"Reward at 50 steps: {total_reward_50}\n"
    logs += f"Reward at 100 steps: {total_reward_100}\n"
    logs += f"Total reward: {total_reward}\n"
    logs += f"Total invalid action: {invalid_count}\n"
    logs += f"Total tokens: {total_tokens}\n"
    logs += f"Total reasoning tokens: {total_reasoning_tokens}\n"
    logs += "-"*200 + "\n"
    logs += f"Final prompt: {prompt}\n"
    log_run(run_name, logs, system_prompt, params, scenario_cfg, log_path)

# Claude & Anthropic SDK

The code above is for GPT models and the Azure OpenAI SDK. In this section we show how we adapt the single-step experiment to use Claude Sonnet 4.6 and the Anthropic SDK.

In [ ]:
from anthropic import AnthropicFoundry

# --------------------------------------------------------------------------------
#                                       SETUP
# --------------------------------------------------------------------------------

# Client setup
endpoint = ""
EcoLogits.init(providers="anthropic", electricity_mix_zone="SWE")
client = AnthropicFoundry(
    api_key='',
    base_url=endpoint
)

# Model setup
model_name = "claude-sonnet-4-6"
temperature = 1
reasoning = None
max_tokens = 1024
params_llm = {
    "model_name": model_name,
    "temperature": temperature,
    "reasoning": reasoning,
    "max_tokens": max_tokens
}

# Prompt setup
prompt_path = "llm_agents/prompts/"
prompt_file = "base-24-strat.md"
with open(prompt_path + prompt_file, "r") as file:
    system_prompt = file.read()

# --------------------------------------------------------------------------------
#                                     EXPERIMENT
# --------------------------------------------------------------------------------

nb_run = 15
num_steps = 100
global_run_name = "prompt1-base-BLine-NoGreen-Sonnet"

for i in range(nb_run):
  print(f"Run number {i + 1}")

  # Run Params
  run_name = f"{global_run_name}-{i + 1}"
  print(run_name)

  # Init useful var
  regex = r'take_action\(([a-zA-Z]+), *(\w*)\)'
  logs = ""
  step = 0
  total_tokens = 0
  total_reasoning_tokens = 0
  total_reward = 0
  total_reward_30 = 0
  total_reward_50 = 0
  total_reward_100 = 0
  invalid_count = 0
  invalid_action = False

  # Init Env & action
  action = "None"
  observation, _ = env.reset()

  for i in tqdm(range(num_steps)):
      # Start turn with log
      step += 1
      logs += f"Step {step}\n"
      logs += f"Observation: {format_obs(observation, labels, decoys)}\n"
      logs += f"True State:\n{true_obs_to_table(cyborg.get_agent_state('True'), cyborg)}\n"

      # Start timer
      start = time.time()

      # Construct prompt
      prompt = [
          {
              "role": "user",
              "content": format_obs(observation, labels, decoys),
          }
      ]

      # Get action from LLM
      if reasoning is None:
          response = client.messages.create(
              system=system_prompt,
              messages=prompt,
              temperature=temperature,
              model=model_name,
              max_tokens = max_tokens,
          )
      else:
          response = client.messages.create(
              system=system_prompt,
              messages=prompt,
              reasoning_effort=reasoning,
              model=model_name,
              max_tokens = max_tokens,
          )

      # Extract action and target
      try:
          regex_match = re.search(regex, response.content[0].text)
          if regex_match:
              action = regex_match.group(1)
              target = regex_match.group(2)
          else:
              raise Exception
      except Exception:
          action = "Sleep"
          target = ""
          invalid_action = True
          invalid_count += 1

      # Handle Decoy
      if action == 'Decoy':
          try:
              action, decoys = get_decoy(decoys, target)
          except Exception:
              action = "Sleep"
              target = ""
              invalid_action = True
              invalid_count += 1
      if action == 'Restore':
          try:
              decoys = reset_decoys(decoys, target)
          except Exception:
              action = "Sleep"
              target = ""
              invalid_action = True
              invalid_count += 1

      # Check validity
      if not is_llm_response_valid(action, target, action_to_int):
          action = "Sleep"
          target = ""
          invalid_action = True
          invalid_count += 1

      # Get action index
      action_idx = get_action_index(action, target, action_to_int)

      # Take step
      observation, reward, _, _, _ = env.step(action_idx)

      # End timer
      end = time.time()

      # Log
      total_reward += reward
      total_tokens += response.usage.input_tokens + response.usage.output_tokens
      if reasoning is not None:
          total_reasoning_tokens += response.usage.completion_tokens_details.reasoning_tokens
      logs += f"Red Action: {env.get_last_action('Red')} ({get_action_res(env.get_observation('Red')['success'].value)})\n"
      if invalid_action:
          logs += f"Blue Action (Invalid): {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
          invalid_action = False
      else:
          logs += f"Blue Action: {env.get_last_action('Blue')} ({get_action_res(env.get_observation('Blue')['success'].value)})\n"
      logs += f"Reward: {reward}\n"
      logs += f"Prompt Tokens: {response.usage.input_tokens}\n"
      logs += f"Response Tokens: {response.usage.output_tokens}\n"
      if reasoning is not None:
          logs += f"Reasoning Tokens: {response.usage.completion_tokens_details.reasoning_tokens}\n"
      logs += f"Response: {response.model_dump_json()}\n"
      logs += f"Time: {end - start} s\n"
      if step == 30:
          total_reward_30 = total_reward
      if step == 50:
          total_reward_50 = total_reward
      if step == 100:
          total_reward_100 = total_reward

      logs += "-"*200 + "\n"

  # Final log
  print(f"Reward at 30 steps: {total_reward_30}")
  print(f"Reward at 50 steps: {total_reward_50}")
  print(f"Reward at 100 steps: {total_reward_100}")
  print(f"Total reward: {total_reward}")
  print(f"Total invalid action: {invalid_count}")
  print(f"Total tokens: {total_tokens}")
  print(f"Total reasoning tokens: {total_reasoning_tokens}")
  logs += f"Reward at 30 steps: {total_reward_30}\n"
  logs += f"Reward at 50 steps: {total_reward_50}\n"
  logs += f"Reward at 100 steps: {total_reward_100}\n"
  logs += f"Total reward: {total_reward}\n"
  logs += f"Total invalid action: {invalid_count}\n"
  logs += f"Total tokens: {total_tokens}\n"
  logs += f"Total reasoning tokens: {total_reasoning_tokens}\n"
  logs += "-"*200 + "\n"
  logs += f"Final prompt: {prompt}\n"
  log_run(run_name, logs, system_prompt, params, scenario_cfg, log_path)

# Cleanup

In [ ]:
client.close()